In [5]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
# from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

In [3]:
LITE_MODE = True

In [6]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


In [7]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [13]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    
    for row in reader:
        human_predictions.append(float(row[1]))

In [14]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [15]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")

Human predicted 120.0 for an item that actually costs 219.0


In [16]:
evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

### Neural Network

In [23]:
# Prepare our documents and prices

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]
print(y[0])

64.3


In [25]:
# Set the random seed for reproducibility
np.random.seed(42)

# Create the HashingVectorizer with specific parameters
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)

# Fit the vectorizer to the documents and transform them into feature matrix X
X = vectorizer.fit_transform(documents)

In [26]:
# Define the neural network - here is Pytorch code to create a 8 layer neural network

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [27]:
# Convert data to PyTorch tensors
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Create the loader
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initialize the model
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [28]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [29]:
# Define loss function and optimizer

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# We will do 2 complete runs through the data

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # The next 4 lines are the 4 stages of training: forward pass, loss calculation, backward pass, optimize
        
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 38151.059, Val Loss: 20336.852


  0%|          | 0/310 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 37665.883, Val Loss: 17355.451


In [30]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [31]:
evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$58 $63 $16 $27 $42 $176 $12 $79 $35 $69 $438 $74 $106 $167 $37 $31 $7 $31 $41 $29 $14 $34 $76 $49 $271 $257 $243 $21 $65 $42 $57 $176 $13 $29 $131 $271 $64 $126 $117 $56 $159 $125 $14 $81 $101 $54 $74 $52 $10 $25 $13 $43 $73 $31 $115 $94 $36 $122 $47 $39 $56 $10 $34 $16 $424 $136 $18 $242 $6 $296 $6 $24 $127 $115 $12 $69 $73 $37 $22 $70 $71 $149 $25 $46 $12 $65 $42 $115 $132 $147 $23 $77 $25 $20 $31 $43 $51 $42 $133 $191 $41 $65 $2 $29 $16 $63 $111 $254 $8 $97 $19 $84 $141 $31 $41 $183 $165 $50 $80 $24 $22 $224 $44 $0 $55 $22 $20 $209 $85 $45 $48 $105 $112 $24 $77 $23 $106 $109 $40 $78 $58 $150 $7 $173 $232 $79 $56 $261 $46 $9 $19 $216 $16 $62 $32 $131 $152 $8 $60 $9 $40 $12 $2 $25 $441 $16 $27 $13 $17 $50 $13 $19 $216 $42 $42 $15 $25 $4 $32 $143 $333 $15 $64 $2 $51 $116 $15 $50 $12 $12 $66 $70 $93 $58 $15 $34 $88 $6 $4 $13 

In [32]:
def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [33]:
print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [34]:
messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [39]:
from openai import OpenAI

base_url = "https://openrouter.ai/api/v1"
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

openai = OpenAI(base_url=base_url, api_key=openrouter_api_key)
def gpt_4__1_nano(item):
    response = openai.chat.completions.create(
        model="openai/gpt-4.1-nano",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content


In [47]:
gpt_4__1_nano(test[0])

'$250'

In [48]:
test[0].price

219.0

In [42]:
evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $20 $120 $80 $6 $65 $11 $870 $363 $20 $30 $9 $29 $8 $71 $5 $40 $31 $54 $26 $65 $75 $182 $254 $705 $5 $251 $64 $30 $15 $10 $50 $35 $119 $90 $31 $36 $13 $175 $45 $20 $105 $70 $0 $32 $13 $65 $52 $22 $105 $225 $10 $197 $16 $8 $110 $52 $13 $86 $28 $41 $30 $29 $30 $90 $295 $25 $74 $16 $8 $30 $6 $25 $20 $26 $0 $3 $3 $30 $4 $15 $74 $11 $10 $32 $44 $0 $4 $13 $20 $5 $20 $2 $78 $11 $7 $20 $425 $50 $3 $11 $11 $1 $32 $10 $370 $14 $49 $10 $136 $89 $48 $54 $80 $1 $4 $24 $47 $19 $311 $80 $16 $50 $10 $5 $51 $29 $94 $79 $13 $65 $5 $85 $0 $55 $10 $78 $62 $16 $50 $70 $12 $44 $118 $5 $690 $15 $13 $3 $144 $2 $10 $3 $129 $101 $41 $30 $5 $311 $17 $6 $2 $240 $7 $752 $30 $5 $5 $10 $3 $70 $8 $52 $101 $3 $57 $4 $23 $546 $25 $150 $1 $30 $8 $73 $17 $20 $2 $15 $99 $15 $11 $55 $70 $10 $30 $21 $1 

In [49]:
def claude_opus_4_5(item):
    response = openai.chat.completions.create(
        model="anthropic/claude-opus-4-5",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content


print(claude_opus_4_5(test[0]))

$199


In [50]:
evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$20 $34 $15 $25 $0 $30 $54 $65 $6 $45 $264 $129 $5 $21 $49 $3 $11 $20 $20 $74 $16 $6 $40 $75 $33 $254 $206 $5 $90 $64 $10 $30 $70 $50 $25 $369 $30 $43 $34 $8 $154 $55 $10 $45 $70 $0 $2 $2 $65 $72 $28 $114 $325 $10 $37 $44 $6 $50 $48 $1 $46 $48 $61 $60 $279 $9 $50 $355 $35 $14 $19 $3 $70 $6 $20 $11 $26 $2 $2 $6 $10 $3 $5 $74 $14 $25 $32 $44 $30 $1 $3 $5 $5 $22 $0 $98 $4 $67 $120 $325 $10 $27 $3 $49 $49 $32 $12 $365 $1 $114 $30 $36 $1 $38 $54 $29 $9 $7 $6 $47 $4 $161 $10 $76 $0 $15 $4 $9 $30 $89 $119 $12 $39 $0 $25 $2 $55 $10 $22 $12 $16 $249 $30 $7 $44 $2 $15 $25 $85 $8 $6 $43 $31 $94 $1 $89 $26 $43 $35 $20 $40 $18 $8 $0 $41 $7 $58 $20 $0 $0 $0 $9 $170 $14 $66 $1 $2 $3 $4 $43 $155 $10 $250 $59 $24 $3 $53 $17 $25 $14 $5 $1 $10 $11 $70 $10 $9 $130 $21 $10 

In [56]:
def gemini_3_pro_preview(item):
    response = openai.chat.completions.create(
        model="google/gemini-2.5-pro-preview-03-25",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content

In [57]:
print(gemini_3_pro_preview(test[0]))

$209


In [66]:
def grok_2_1(item):
    response = openai.chat.completions.create(
        model="x-ai/grok-3-beta",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content

In [67]:
print(grok_2_1(test[0]))


$229


In [68]:
test[0].price

219.0

In [76]:
def gpt_5_1(item):
    response = openai.chat.completions.create(
        model="openai/gpt-4.1",
        messages=messages_for(item),
        max_tokens=100,
        temperature=0.0
    )
    return response.choices[0].message.content

In [77]:
print(gpt_5_1(test[0]))

$199
